# Reinforcement Learning: ADP, Q-learning, and Policy Search

ADP (Adaptive Dynamic Programming), Q-learning, and Policy Search are three fundamental approaches in reinforcement learning (RL) that enable agents to learn optimal behaviors through interaction with their environment.

Q-learning is a model-free reinforcement learning algorithm that seeks to learn the value of taking a particular action in a given state. It does this by maintaining a Q-table, which is updated iteratively based on the agent's experiences. The update rule for Q-learning is given by:
```
Q(s, a) ← Q(s, a) + α [r + γ max_a Q(s', a') - Q(s, a)]
```
Where:
- `Q(s, a)` is the current estimate of the value of taking action `a` in state `s`.
- `α` is the learning rate, which determines how much new information overrides old information.
- `r` is the reward received after taking action `a` in state `s`.
- `γ` is the discount factor, which determines the importance of future rewards
- `max_a Q(s', a')` is the maximum estimated value of the next state

Policy Search, on the other hand, focuses on directly optimizing the policy that the agent follows. Instead of learning value functions, policy search methods aim to find the best policy by adjusting the parameters of a policy function. This can be done using various optimization techniques, such as gradient ascent or evolutionary algorithms. The goal is to maximize the expected cumulative reward over time.

## Environment: Package Delivery World

A robot moves between two locations: Store and Home.

The package status can be:

- waiting: the package is waiting at the Store;
- carrying: the robot has picked up the package;
- delivered: the package has been delivered to Home.

State = (location, package_status)

Actions:

- GoStore
- GoHome
- PickUp
- Deliver
- Wait

Rewards:

- Valid PickUp at the Store when the package is waiting: +5
- Valid Deliver at Home when carrying the package: +20
- Invalid pickup or delivery: -3
- Movement and waiting: -1
- Terminal delivered states: 0

Movement succeeds with probability 0.90; otherwise the robot stays in the same location.



In [50]:
from collections import Counter, defaultdict
from itertools import product
import random
import statistics

LOCATIONS = ("Store", "Home")
PACKAGE_STATUSES = ("waiting", "carrying", "delivered")
ACTIONS = ("GoStore", "GoHome", "PickUp", "Deliver", "Wait")
GAMMA = 0.90
MOVE_SUCCESS_PROBABILITY = 0.90
MAX_STEPS = 15

STATES = tuple((location, status) for location in LOCATIONS for status in PACKAGE_STATUSES)
NONTERMINAL_STATES = tuple(state for state in STATES if state[1] != "delivered")


def is_terminal(state):
    return state[1] == "delivered"


def reset_state(rng):
    return (rng.choice(LOCATIONS), "waiting")


def state_label(state):
    location, status = state
    return f"location={location}, package={status}"


print("States:", len(STATES))
print("Non-terminal states:", len(NONTERMINAL_STATES))
print("Actions:", ACTIONS)

States: 6
Non-terminal states: 4
Actions: ('GoStore', 'GoHome', 'PickUp', 'Deliver', 'Wait')


## Environment step function

This function is shared by all three learning methods as the environment step function. It takes the current state and an action as input, and returns the next state, reward, and a boolean indicating whether the episode has ended.

In [ ]:
def delivery_step(state, action, rng):
    location, status = state

    if is_terminal(state):
        return state, 0.0

    if action == "GoStore":
        reward = -1.0
        if rng.random() < MOVE_SUCCESS_PROBABILITY:
            next_state = ("Store", status)
        else:
            next_state = state

    elif action == "GoHome":
        reward = -1.0
        if rng.random() < MOVE_SUCCESS_PROBABILITY:
            next_state = ("Home", status)
        else:
            next_state = state

    elif action == "PickUp":
        if location == "Store" and status == "waiting":
            next_state = (location, "carrying")
            reward = 5.0
        else:
            next_state = state
            reward = -3.0

    elif action == "Deliver":
        if location == "Home" and status == "carrying":
            next_state = (location, "delivered")
            reward = 20.0
        else:
            next_state = state
            reward = -3.0

    elif action == "Wait":
        next_state = state
        reward = -0.02

    else:
        raise ValueError(f"Unknown action: {action}")

    return next_state, reward

## Common evaluation helpers

You will use the same evaluation functions for ADP, Q-learning, and policy search.



In [53]:
def run_episode(policy, start_state=("Store", "waiting"), seed=0, max_steps=MAX_STEPS, return_trace=False):
    rng = random.Random(seed)
    state = start_state
    total_return = 0.0
    discount = 1.0
    trace = []

    for _ in range(max_steps):
        if is_terminal(state):
            break
        action = policy(state)
        next_state, reward = delivery_step(state, action, rng)
        trace.append((state, action, reward, next_state))
        total_return += discount * reward
        discount *= GAMMA
        state = next_state

    if return_trace:
        return total_return, trace
    return total_return


def evaluate_policy(policy, episodes=100, seed=100):
    rng = random.Random(seed)
    returns = []
    for episode_index in range(episodes):
        start_state = reset_state(rng)
        returns.append(run_episode(policy, start_state=start_state, seed=seed + episode_index))
    return statistics.mean(returns)


def print_policy(policy_table, title="Policy"):
    print(title)
    for state in NONTERMINAL_STATES:
        print(f"  {state_label(state):36s} -> {policy_table.get(state, '?')}")

ADP is model-based. You will learn an empirical model from experience:

- transition counts for estimating P(s_next | s, a);
- average observed rewards for estimating R(s, a, s_next).

Then you will solve the learned model with value iteration.



In [ ]:
def collect_random_experience(episodes=300, seed=1):
    rng = random.Random(seed)
    transition_counts = defaultdict(Counter)
    reward_sums = defaultdict(float)

    for episode_index in range(episodes):
        state = reset_state(rng)
        for _ in range(MAX_STEPS):
            if is_terminal(state):
                break

            # choose a random action, step in the environment,
            # and update transition_counts and reward_sums.
            # Suggested keys:
            # transition_counts[(state, action)][next_state] += 1
            # reward_sums[(state, action, next_state)] += reward
            rng_action = rng.choice(ACTIONS)
            next_state, reward = delivery_step(state, rng_action, rng)
            transition_counts[(state, rng_action)][next_state] += 1
            reward_sums[(state, rng_action, next_state)] += reward
            state = next_state
            # pass

    return transition_counts, reward_sums


transition_counts, reward_sums = collect_random_experience()
print("Observed state-action pairs:", len(transition_counts))

Observed state-action pairs: 20


Convert counts into probabilities using maximum likelihood estimation. This means that the probability of transitioning to a next state given a current state and action is estimated as the number of times that transition was observed divided by the total number of times that action was taken in that state.

In [ ]:
def build_model(transition_counts, reward_sums):
    transition_model = {}
    reward_model = {}

    for state_action, next_counter in transition_counts.items():
        # convert counts into probabilities.
        total_count = sum(next_counter.values())
        transition_model[state_action] = {}

        for next_state, count in next_counter.items():
            probability = count / total_count if total_count > 0 else 0
            transition_model[state_action][next_state] = probability

            # compute the average reward for this transition.
            reward_model[(state_action[0], state_action[1], next_state)] = None
            reward = reward_sums.get((state_action[0], state_action[1], next_state), 0.0)
            average_reward = reward / count if count > 0 else 0.0
            reward_model[(state_action[0], state_action[1], next_state)] = average_reward

    return transition_model, reward_model


transition_model, reward_model = build_model(transition_counts, reward_sums)
print("Learned transition-model entries:", len(transition_model))

Learned transition-model entries: 20


For each non-terminal state, value iteration uses the best action under the learned model:

$$V(s) = \max_a \sum_{s'} P(s'|s, a) \left[ R(s, a, s') + \gamma V(s') \right] $$


In [ ]:
def model_q_value(state, action, values, transition_model, reward_model, gamma=GAMMA):
    if (state, action) not in transition_model:
        # Unknown actions are discouraged but not impossible.
        return -5.0 + gamma * values[state]

    total = 0.0
    # compute expected action utility from the learned model.
    for next_state, probability in transition_model[(state, action)].items():
        reward = reward_model.get((state, action, next_state), 0.0)
        total += probability * (reward + gamma * values[next_state])
    return total


def value_iteration_from_model(transition_model, reward_model, iterations=80):
    values = {state: 0.0 for state in STATES}

    for _ in range(iterations):
        new_values = values.copy()
        for state in NONTERMINAL_STATES:
            # Bellman optimality backup using model_q_value.
            new_values[state] = max(model_q_value(state, action, values, 
                                                  transition_model, reward_model) for action in ACTIONS)
        values = new_values

    policy = {}
    for state in NONTERMINAL_STATES:
        # choose the action with the largest model_q_value.
        policy[state] = max(ACTIONS, key=lambda action: model_q_value(state, action, values, 
                                                                      transition_model, reward_model))

    return values, policy


adp_values, adp_policy = value_iteration_from_model(transition_model, reward_model)
print_policy(adp_policy, "ADP policy")
print("ADP mean return:", evaluate_policy(lambda state: adp_policy.get(state, "Wait")))

ADP policy
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
ADP mean return: 18.488177000000004


Q-learning is model-free. It learns Q(s, a) directly from transitions.

Action-Utility Learning with Q-learning means that the agent learns the expected utility of taking a specific action in a given state, without needing to know the underlying model of the environment. The agent updates its Q-values based on the rewards received and the estimated future rewards, allowing it to improve its policy over time.

The update is:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$


In [ ]:
def epsilon_greedy_action(q_values, state, epsilon, rng):
    # explore randomly with probability epsilon;
    # otherwise choose the action with the highest Q-value.
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)
    else:
        return max(ACTIONS, key=lambda action: q_values[(state, action)])
    raise NotImplementedError("TODO: implement epsilon_greedy_action")


def q_learning(episodes=800, alpha=0.25, seed=2):
    rng = random.Random(seed)
    q_values = defaultdict(float)
    scores = []

    for episode_index in range(episodes):
        state = reset_state(rng)
        epsilon = max(0.05, 0.50 * (1 - episode_index / episodes))

        for _ in range(MAX_STEPS):
            if is_terminal(state):
                break

            action = epsilon_greedy_action(q_values, state, epsilon, rng)
            next_state, reward = delivery_step(state, action, rng)

            # compute the Q-learning target and update q_values[(state, action)].
            best_next_q = max(q_values[(next_state, a)] for a in ACTIONS)
            q_values[(state, action)] += alpha * (reward + GAMMA * best_next_q - q_values[(state, action)])
            state = next_state

        if episode_index in {0, 9, 49, 199, episodes - 1}:
            policy = {state: max(ACTIONS, key=lambda action: q_values[(state, action)]) for state in NONTERMINAL_STATES}
            scores.append((episode_index + 1, evaluate_policy(lambda s, p=policy: p.get(s, "Wait"), episodes=30, seed=300)))

    policy = {state: max(ACTIONS, key=lambda action: q_values[(state, action)]) for state in NONTERMINAL_STATES}
    return q_values, policy, scores


q_values, q_policy, q_scores = q_learning()
print("Q-learning checkpoints:")
for episode, score in q_scores:
    print(f"  after {episode:>3} episodes: mean return = {score:+.3f}")
print_policy(q_policy, "Q-learning policy")
print("Q-learning mean return:", evaluate_policy(lambda state: q_policy.get(state, "Wait")))

Q-learning checkpoints:
  after   1 episodes: mean return = +4.170
  after  10 episodes: mean return = +18.732
  after  50 episodes: mean return = +18.732
  after 200 episodes: mean return = +18.732
  after 800 episodes: mean return = +18.732
Q-learning policy
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
Q-learning mean return: 18.488177000000004


Policy search directly searches over policies. Because this environment has only four non-terminal states and five actions, a deterministic table policy has $5^4 = 625$ possibilities.

This exhaustive search is small enough for practice, but the idea does not scale to large environments without smarter search methods.



In [ ]:
def policy_from_action_tuple(action_tuple):
    # map each non-terminal state to one action in action_tuple.
    return {state: action for state, action in zip(NONTERMINAL_STATES, action_tuple)}
    raise NotImplementedError("TODO: implement policy_from_action_tuple")


def exhaustive_policy_search():
    best_policy = None
    best_score = float("-inf")
    checked = 0

    for action_tuple in product(ACTIONS, repeat=len(NONTERMINAL_STATES)):
        checked += 1
        policy_table = policy_from_action_tuple(action_tuple)

        # evaluate policy_table and keep the best one.
        score = evaluate_policy(lambda state: policy_table.get(state, "Wait"), episodes=30, seed=400)
        if score > best_score:
            best_score = score
            best_policy = policy_table

    return best_policy, best_score, checked


search_policy, search_score, checked = exhaustive_policy_search()
print("Policies checked:", checked)
print_policy(search_policy, "Best policy found by policy search")
print("Policy-search mean return:", search_score)

Policies checked: 625
Best policy found by policy search
  location=Store, package=waiting      -> PickUp
  location=Store, package=carrying     -> GoHome
  location=Home, package=waiting       -> GoStore
  location=Home, package=carrying      -> Deliver
Policy-search mean return: 18.621200000000005


In [64]:
# Optional comparison cell. Run after finishing all parts.
methods = {
    "ADP": lambda state: adp_policy.get(state, "Wait"),
    "Q-learning": lambda state: q_policy.get(state, "Wait"),
    "Policy search": lambda state: search_policy.get(state, "Wait"),
}

for name, policy in methods.items():
    print(f"{name:>13}: {evaluate_policy(policy, episodes=200, seed=999):+.3f}")

print("\nSample traces from (Store, waiting):")
for name, policy in methods.items():
    total, trace = run_episode(policy, start_state=("Store", "waiting"), seed=42, return_trace=True)
    print(f"\n{name}: return={total:+.3f}")
    for state, action, reward, next_state in trace:
        print(f"  {state_label(state):36s} --{action:>7}/{reward:+.1f}--> {state_label(next_state)}")

          ADP: +18.457
   Q-learning: +18.457
Policy search: +18.457

Sample traces from (Store, waiting):

ADP: return=+20.300
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/-1.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered

Q-learning: return=+20.300
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/-1.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered

Policy search: return=+20.300
  location=Store, package=waiting      -- PickUp/+5.0--> location=Store, package=carrying
  location=Store, package=carrying     -- GoHome/-1.0--> location=Home, package=carrying
  location=Home, package=carrying      --Deliver/+20.0--> location=Home, package=delivered
